In [1]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 1}
import datetime as dt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

import warnings
warnings.simplefilter(action="ignore")

pd.set_option('display.max_columns',1000)
pd.set_option('display.width', 500)
pd.set_option('display.float_format',lambda x : '%.2f' % x)

In [2]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 2}
df_ = pd.read_csv("data/dataset.csv", compression="gzip")
df = df_.copy()
df.head()

,RecipeId,Name,CookTime,PrepTime,TotalTime,RecipeIngredientParts,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeInstructions
0,38,Low-Fat Berry Blue Frozen Dessert,1440,45,1485,"c(""blueberries"", ""granulated sugar"", ""vanilla ...",170.90,2.50,1.30,8.00,29.80,37.10,3.60,30.20,3.20,"c(""Toss 2 cups berries with sugar."", ""Let stan..."
1,41,Carina's Tofu-Vegetable Kebabs,20,1440,1460,"c(""extra firm tofu"", ""eggplant"", ""zucchini"", ""...",536.10,24.00,3.80,0.00,1558.60,64.20,17.30,32.10,29.30,"c(""Drain the tofu, carefully squeezing out exc..."
2,42,Cabbage Soup,30,20,50,"c(""plain tomato juice"", ""cabbage"", ""onion"", ""c...",103.60,0.40,0.10,0.00,959.30,25.10,4.80,17.70,4.30,"c(""Mix everything together and bring to a boil..."
3,45,Buttermilk Pie With Gingersnap Crumb Crust,50,30,80,"c(""sugar"", ""margarine"", ""egg"", ""flour"", ""salt""...",228.00,7.10,1.70,24.50,281.80,37.50,0.50,24.70,4.20,"c(""Preheat oven to 350°F."", ""Make pie crust, u..."
4,46,A Jad - Cucumber Pickle,0,25,25,"c(""rice vinegar"", ""haeo"")",4.30,0.00,0.00,0.00,0.70,1.10,0.20,0.20,0.10,"c(""Slice the cucumber in four lengthwise, then..."


In [3]:
# --- [CELL 2]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 3}
def grab_col_names(dataframe, cat_th=10, car_th=20):

    cat_cols = [col for col in dataframe.columns if dataframe[col].dtypes == "O"]
    num_but_cat = [col for col in dataframe.columns if dataframe[col].nunique() < cat_th and
                   dataframe[col].dtypes != "O"]
    cat_but_car = [col for col in dataframe.columns if dataframe[col].nunique() > car_th and
                   dataframe[col].dtypes == "O"]
    cat_cols = cat_cols + num_but_cat
    cat_cols = [col for col in cat_cols if col not in cat_but_car]

    # num_cols
    num_cols = [col for col in dataframe.columns if dataframe[col].dtypes != "O"]
    num_cols = [col for col in num_cols if col not in num_but_cat]

    print(f"Observations: {dataframe.shape[0]}")
    print(f"Variables: {dataframe.shape[1]}")
    print(f'cat_cols: {len(cat_cols)}')
    print(f'num_cols: {len(num_cols)}')
    print(f'cat_but_car: {len(cat_but_car)}')
    print(f'num_but_cat: {len(num_but_cat)}')
    return cat_cols, num_cols, cat_but_car

In [4]:
# --- [CELL 3]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 4}
cat_cols, num_cols, num_but_cat = grab_col_names(df)

Observations: 375703
Variables: 16
cat_cols: 0
num_cols: 13
cat_but_car: 3
num_but_cat: 0


In [5]:
# --- [CELL 4]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 5}
def outlier_thresholds(dataframe, col_name, q1=0.01, q3=0.99):
    quartile1= dataframe[col_name].quantile(q1)
    quartile3= dataframe[col_name].quantile(q3)
    interquantile_range = quartile3 -quartile1
    up_limit= quartile3 +1.5 * interquantile_range
    low_limit= quartile1 -1.5 * interquantile_range
    return low_limit, up_limit

In [6]:
# --- [CELL 5]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 6}
def replace_with_thresholds(dataframe, variable):
    low_limit, up_limit = outlier_thresholds(dataframe, variable)
    dataframe.loc[(dataframe[variable] < low_limit), variable] = low_limit
    dataframe.loc[(dataframe[variable] > up_limit), variable] = up_limit

for col in num_cols:
    replace_with_thresholds(df, col)

In [7]:
# --- [CELL 6]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 7}
def check_outlier(dataframe, col_name):
    low_limit, up_limit = outlier_thresholds(dataframe, col_name)
    if dataframe[(dataframe[col_name] > up_limit) | (dataframe[col_name] < low_limit)].any(axis=None):
        return True
    else:
        return False

check_outlier(df,num_cols)

False

In [8]:
# --- [CELL 7]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 8}
df= df.iloc[:,1:]

In [9]:
# --- [CELL 8]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 9}
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from yellowbrick.cluster import KElbowVisualizer
from scipy.cluster.hierarchy import linkage
from scipy.cluster.hierarchy import dendrogram
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import AgglomerativeClustering

In [10]:
# --- [CELL 9]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 10}
cat_cols, num_cols, num_but_cat = grab_col_names(df)

Observations: 375703
Variables: 15
cat_cols: 0
num_cols: 12
cat_but_car: 3
num_but_cat: 0


In [11]:
# --- [CELL 10]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 11}
df2=df.copy()

In [12]:
# --- [CELL 11]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 12}
sc = MinMaxScaler((0, 1))
df2[num_cols] = sc.fit_transform(df2[num_cols])

In [13]:
# --- [CELL 12]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 13}
kmeans = KMeans(n_clusters=30, n_init="auto").fit(df2[["TotalTime","Calories","SugarContent"]])

In [14]:
# --- [CELL 13]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 14}
clusters_kmeans = kmeans.labels_
clusters_kmeans

array([19, 19, 27, ...,  7,  1, 10], dtype=int32)

In [15]:
# --- [CELL 14]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 15}
df["kmeans_cluster"] = clusters_kmeans
df["kmeans_cluster"]= df["kmeans_cluster"] + 1
df.head()

,Name,CookTime,PrepTime,TotalTime,RecipeIngredientParts,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeInstructions,kmeans_cluster
0,Low-Fat Berry Blue Frozen Dessert,1200,45,1485.00,"c(""blueberries"", ""granulated sugar"", ""vanilla ...",170.90,2.50,1.30,8.00,29.80,37.10,3.60,30.20,3.20,"c(""Toss 2 cups berries with sugar."", ""Let stan...",20
1,Carina's Tofu-Vegetable Kebabs,20,600,1460.00,"c(""extra firm tofu"", ""eggplant"", ""zucchini"", ""...",536.10,24.00,3.80,0.00,1558.60,64.20,17.30,32.10,29.30,"c(""Drain the tofu, carefully squeezing out exc...",20
2,Cabbage Soup,30,20,50.00,"c(""plain tomato juice"", ""cabbage"", ""onion"", ""c...",103.60,0.40,0.10,0.00,959.30,25.10,4.80,17.70,4.30,"c(""Mix everything together and bring to a boil...",28
3,Buttermilk Pie With Gingersnap Crumb Crust,50,30,80.00,"c(""sugar"", ""margarine"", ""egg"", ""flour"", ""salt""...",228.00,7.10,1.70,24.50,281.80,37.50,0.50,24.70,4.20,"c(""Preheat oven to 350°F."", ""Make pie crust, u...",8
4,A Jad - Cucumber Pickle,0,25,25.00,"c(""rice vinegar"", ""haeo"")",4.30,0.00,0.00,0.00,0.70,1.10,0.20,0.20,0.10,"c(""Slice the cucumber in four lengthwise, then...",11


In [16]:
# --- [CELL 15]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 16}
# === BEFORE (original) ===
# df.groupby('kmeans_cluster').agg({1: ['count','mean', 'median', 'sum'],
#                                     2: ['count','mean', 'median', 'sum'],
#                                     3: ['count','mean', 'median', 'sum'],
#                                     4: ['count','mean','median', 'sum']})

# === AFTER (edited) ===
df.groupby('kmeans_cluster').agg({
    "TotalTime": ['count', 'mean', 'median', 'sum'],
    "Calories": ['count', 'mean', 'median', 'sum'],
    "SugarContent": ['count', 'mean', 'median', 'sum']
})

TotalTime                            Calories                           SugarContent                       
                   count    mean  median        sum    count    mean median        sum        count  mean median       sum
kmeans_cluster                                                                                                            
1                  22000   40.10   30.00  882263.00    22000  124.19 120.70 2732113.70        22000  6.65   6.60 146203.10
2                  12960   47.50   40.00  615635.00    12960  242.11 241.90 3137762.60        12960 21.79  21.70 282397.50
3                  37476   37.85   30.00 1418311.00    37476  201.90 200.70 7566534.00        37476  1.29   1.30  48450.50
4                   9979   55.07   45.00  549540.00     9979  305.21 309.10 3045662.40         9979 31.54  31.50 314773.00
5                    671 1569.08 1470.00 1052852.50      671  282.22 231.00  189372.60          671 15.17  14.70  10177.20
6                  16873   55.87   45.00  942658.00    16873  330.09 325.90 5569686.80        16873  8.67   8.60 146213.30
7                  18230   55.59   40.00 1013404.00    18230  462.59 454.40 8432951.00        18230  5.64   5.60 102760.20
8                  12745   50.15   40.00  639138.00    12745  286.86 285.40 3656052.40        12745 26.31  26.20 335261.20
9                  16952   42.23   30.00  715800.00    16952  169.47 168.50 2872808.00        16952 13.54  13.50 229554.60
10                  1423  475.93  480.00  677248.00     1423  323.26 305.40  459999.00         1423 24.25  24.00  34504.60
11                 35460   29.78   20.00 1055921.00    35460   61.46  61.90 2179355.10        35460  0.68   0.50  24140.50
12                  2285   69.82   50.00  159540.00     2285  733.72 672.40 1676555.70         2285 29.69  29.20  67849.60
13                  5105   62.29   45.00  318008.00     5105  572.63 554.30 2923296.40         5105 19.29  19.10  98498.30
14                  1705   74.45   45.00  126929.00     1705 1027.37 963.00 1751660.40         1705 11.89  11.70  20264.60
15                 27672   45.93   40.00 1270992.00    27672  273.63 270.60 7571836.70        27672  4.58   4.50 126822.70
16                 18054   41.87   30.00  755835.00    18054  141.44 138.60 2553583.40        18054  9.93   9.90 179365.00
17                  4684  527.09  495.00 2468889.00     4684  320.33 321.50 1500443.20         4684  3.47   3.40  16251.90
18                  9432   54.83   40.00  517146.00     9432  400.38 391.80 3776371.90         9432 13.70  13.60 129206.70
19                 28354   45.40   35.00 1287150.00    28354  346.58 344.70 9826895.80        28354  1.81   1.90  51190.50
20                   424 1569.35 1470.00  665405.00      424  351.70 284.50  149120.10          424 30.00  29.40  12721.60
21                  6503  267.50  252.00 1739533.00     6503  176.19 171.70 1145795.60         6503  2.83   2.70  18376.60
22                  8551   64.11   50.00  548216.00     8551  366.98 365.90 3138064.40         8551 36.91  36.90 315628.80
23                   804 1827.90 1834.50 1469629.50      804  212.51 142.90  170856.70          804  2.91   2.00   2341.20
24                  7164   57.40   40.00  411180.00     7164  756.76 730.75 5421411.00         7164  4.31   4.40  30853.10
25                 26917   35.42   30.00  953284.00    26917  103.31 104.10 2780769.70        26917  3.55   3.50  95580.30
26                  1272 1438.93 1455.00 1830323.00     1272  234.94 188.55  298847.10         1272  3.06   2.40   3895.30
27                  2940  448.51  435.00 1318620.00     2940  341.77 335.65 1004814.80         2940 11.34  10.90  33327.60
28                 15870   46.81   35.00  742833.00    15870  209.00 207.00 3316875.90        15870 17.63  17.60 279843.20
29                 14877   53.21   40.00  791633.00    14877  516.30 505.30 7680963.50        14877  2.15   2.30  32009.00
30                  8321   61.70   45.00  513394.00     8321  601.18 593.10 500

In [17]:
numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
assert 'kmeans_cluster' in numeric_columns, 'Expected kmeans_cluster to be a numeric grouping column.'
numeric_columns.remove('kmeans_cluster')
assert len(numeric_columns) >= 5, 'Expected at least five numeric feature columns for this aggregation test.'

agg_result = df.groupby('kmeans_cluster').agg({
    numeric_columns[1]: ['count', 'mean', 'median', 'sum'],
    numeric_columns[2]: ['count', 'mean', 'median', 'sum'],
    numeric_columns[3]: ['count', 'mean', 'median', 'sum'],
    numeric_columns[4]: ['count', 'mean', 'median', 'sum'],
})
assert not agg_result.empty, 'Grouped aggregation should produce a non-empty result.'
assert set(['count', 'mean', 'median', 'sum']).issubset(set(agg_result.columns.get_level_values(1)))

try:
    df.groupby('kmeans_cluster').agg({
        1: ['count', 'mean', 'median', 'sum'],
        2: ['count', 'mean', 'median', 'sum'],
        3: ['count', 'mean', 'median', 'sum'],
        4: ['count', 'mean', 'median', 'sum'],
    })
except KeyError:
    pass
else:
    raise AssertionError('Bug regression: integer-labeled aggregation keys unexpectedly succeeded.')